# SlideRule: on-demand ICESat-2 and GEDI processing

[SlideRule](https://slideruleearth.io/) processes satellite altimetry and
lidar data in the cloud. `ea.plugins.sliderule` sends a processor, its
parameters and an area of interest, and returns geolocated points as a
`GeoDataFrame`. SlideRule manages its own credentials.

## Setup

```bash
pip install earthaccess geopandas matplotlib panel holoviews datashader
pip install "earthaccess-plugin-sliderule @ git+https://github.com/betolink/earthaccess-plugin-sliderule"
```

Register at https://slideruleearth.io and use your organization below.

In [ ]:
import earthaccess as ea

ea.login()
ea.plugins.sliderule.configure(url="slideruleearth.io", organization="sliderule")
ea.list_plugins(details=True)

## Search

A small ATL03 (ICESat-2) region over Greenland.

In [ ]:
bbox = (-43.5, 66.5, -43.3, 66.7)  # a small ATL03 region

atl03 = ea.search_data(
    short_name="ATL03",
    bounding_box=bbox,
    temporal=("2023-06-01", "2023-06-30"),
    count=1,
)
len(atl03)

## What SlideRule can do

The plugin reports the processors it can run and the subsetting it does:

In [ ]:
ea.plugins.sliderule.capabilities("ATL03")

## Process with `atl03x`

`request(...).results()` sends the request and returns a `GeoDataFrame`. The searched
granules scope the request; the `aoi` restricts the region.

In [ ]:
aoi = {
    "type": "Polygon",
    "coordinates": [
        [[-43.5, 66.5], [-43.3, 66.5], [-43.3, 66.7], [-43.5, 66.7], [-43.5, 66.5]]
    ],
}

df = ea.plugins.sliderule.request(atl03, api="atl03x", parms={}, aoi=aoi).results()
df.shape

In [ ]:
df.head()

In [ ]:
df.plot(column="height", markersize=0.5)  # geolocated points from SlideRule

## Photon returns: along-track profile

`atl03x` can also classify every photon with the ATL08 algorithm. Passing
`atl08_class` adds the return type (`0` noise, `1` ground, `2` canopy,
`3` top of canopy, `4` unclassified), enough to look at vegetation structure
along a beam.

We pull two ATL03 granules over Boulder, Colorado and cache the returned
`GeoDataFrame` as GeoParquet, so re-running the notebook reuses the local file
instead of re-submitting the SlideRule request.

In [ ]:
from pathlib import Path

import geopandas as gpd

# Boulder, Colorado
boulder_bbox = (-105.35, 39.95, -105.15, 40.10)  # west, south, east, north
boulder_aoi = {
    "type": "Polygon",
    "coordinates": [
        [
            [-105.35, 39.95],
            [-105.15, 39.95],
            [-105.15, 40.10],
            [-105.35, 40.10],
            [-105.35, 39.95],
        ]
    ],
}

cache = Path("boulder_photons.parquet")

if cache.exists():
    photons = gpd.read_parquet(cache)  # GeoParquet cache
else:
    atl03_boulder = ea.search_data(
        short_name="ATL03",
        bounding_box=boulder_bbox,
        temporal=("2023-06-01", "2023-08-31"),
        count=2,
    )
    parms_atl08 = {
        # Ask only for classified photons. The 'unclassified' class is mostly
        # possible-TEP noise that would otherwise bury the signal returns.
        "atl08_class": [
            "atl08_noise",
            "atl08_ground",
            "atl08_canopy",
            "atl08_top_of_canopy",
        ],
    }
    photons = ea.plugins.sliderule.request(
        atl03_boulder, api="atl03x", parms=parms_atl08, aoi=boulder_aoi
    ).results()
    photons.to_parquet(cache)  # GeoParquet cache for the next run

photons.shape

The returned `GeoDataFrame` has `x_atc` (along-track distance), `height` and
`atl08_class`. Below is a single interactive along-track profile: along-track
distance in kilometres on the X axis, height in metres on the Y axis, and every
photon coloured by its ATL08 return type. The slider resamples the photons.

In [ ]:
import holoviews as hv
import panel as pn

hv.extension("bokeh")
pn.extension()

return_types = {
    0: "noise",
    1: "ground",
    2: "canopy",
    3: "top of canopy",
}
colors = {
    "noise": "#c7c7c7",
    "canopy": "#1b9e77",
    "top of canopy": "#a6d854",
    "ground": "#e6550d",
}
# Draw order matters: later layers sit on top. Ground is drawn last so it stays
# visible where it overlaps the canopy.
order = ["noise", "canopy", "top of canopy", "ground"]

returns = photons.assign(
    return_type=photons["atl08_class"].map(return_types),
    along_track_km=photons["x_atc"] / 1000,
).dropna(subset=["return_type"])


def profile(percent):
    """One interactive along-track photon profile for a random `percent` sample."""
    sample = returns.sample(frac=percent / 100, random_state=0)
    layers = [
        hv.Scatter(
            sample.loc[
                sample["return_type"] == name, ["along_track_km", "height"]
            ],
            kdims="along_track_km",
            vdims="height",
            label=name,
        ).opts(color=colors[name], size=3, alpha=0.7)
        for name in order
        if (sample["return_type"] == name).any()
    ]
    y0, y1 = sample["height"].quantile([0.005, 0.995])
    return hv.Overlay(layers).opts(
        title=f"Boulder along-track photon profile — {len(sample):,} photons",
        xlabel="Along-track distance (km)",
        ylabel="Height (m)",
        width=1000,
        height=200,
        ylim=(y0, y1),
        legend_position="right",
    )


sample_slider = pn.widgets.DiscreteSlider(
    name="Sample (%)", options=[1, 2, 5, 10, 25, 50, 100], value=25
)

pn.Column(sample_slider, pn.bind(profile, sample_slider))

## Other processors and parameters

Pass any SlideRule API and its parameters. ICESat-2 APIs include `atl03sp`,
`atl06p`, `atl06sp`, `atl08p`, `atl13sp`, `atl24v`; GEDI APIs include
`gedil1b`, `gedil2a`, `gedil4a`. `parms` is passed through unchanged:

```python
segments = ea.plugins.sliderule.request(
    atl03, api="atl06p", parms={"pass_invalid": False}, aoi=aoi,
).results()
```

You can also run from an `aoi` + `parms` alone, with no search results.

In [ ]:
segments = ea.plugins.sliderule.request(
    atl03, api="atl06p", parms={}, aoi=aoi
).results()
segments.shape

## Unsupported datasets fail early

SlideRule only handles ICESat-2 and GEDI; anything else raises before the
backend is contacted:

In [ ]:
from earthaccess.plugins import UnsupportedCollectionError

mursst = ea.search_data(short_name="MUR-JPL-L4-GLOB-v4.1", count=1)
try:
    ea.plugins.sliderule.request(mursst, api="atl06p").results()
except UnsupportedCollectionError as exc:
    print(exc)

## Which plugins accept a dataset?

In [ ]:
atl03.list_services()

## Summary

- `ea.plugins.sliderule.request(results, api=..., parms=..., aoi=...).results()`
  → `GeoDataFrame`.
- SlideRule keeps its own credentials (`configure(...)`).
- Supports only ICESat-2 and GEDI collections; others fail early.